##  RAG （not changing the parameters of model ） 

Search the related document as the context, then LLM answers the question based on the selected document. 
- What can be the document, the new knowledge, private documents after the model trained 
- Reason:  
    - update the knowledge 
    - Halluciation 
    - private documents from the company or local database 


### Steps
1. Document processing:
    - The documents needs to be able to search, chuck the data. .. 
2. Embedding: similar meaning embedding distance small 
    - Query embedding 
    - Document embedding 

3. Vector database: the embedding data needs to store in the local database, which support quick search (similarity search, top-k , metadata search ... )
    - Vector database: ChromaDB stores the embedding + text + metadata. **Good to use with LangChain / LlamaIndex**
    - Better than SQL? 
    - searching methods: 
        - top-k 

----  


4. Retrieval: with the query embedding , find the most similar a few (top-k) chunks from the vector database  

    - similarity search : Between the query embedding and the document embeddding. 

        - cosine similarity 

        - inner product 

        - L2 distance 


    - top-k retrival 



    - reranking 


    - Hybrid search 

----  

5. Prompt construction: get the context, it needs to organized into the prompt 

    - system prompt 

    - user question

    - retreived context  
      
        - template: 
            f""" You are a helpful assistant.
            Answer the question only based on the context below.
            If the answer is not in the context, say you do not know.

            Context:
            {doc1}


            Question:
            {query}
            ... """

    - prompt template 

    - context window constraint 

----  

6. Generation : 


    - API 


    - Transformer fined tuned pretrained model (local )


    - Configuration


----  
7. Evaluation: How is the answer evaluated, how their relevant, the precision, recall . 


----  

8. Code pipeline. 
    - Data prepare: Document loader ---> Text split into chunks ---> embedding  
        - docs = load_documents () 
        - chunks = split_documents(docs) 
        - chunk_embedding= embedding_model.encode (chunks) 

    - Vector store: using the chromadb , Faiss to store these embedding.  index is : 
        - vectordb.add(chunks, embedding= chunk_embedding, metadata = )


    - **Retriever** : Similarity search to find the related message. 
        - question = "what is NLP"
        - query_vec = embedding_model.encode(query_vec)
        - retreived_chunks = vectordb.similarity_search(question , top_k = 3)

    - Template input to the LLM.  Input is the querying and the retrieved  message. 
        - template: prompt = f"""
                            Answer based on the following context only:

                            Context:
                            {retrieved_chunks}

                            Question:
                            {question}
                            """ 


    - **Generator Pipeline**: 

        - answer= LLM.generate(prompt)

 ----     

9. Difference in API and Local model 

    - API: 
        - Generator : OpenAI, Gemini
        - Embedding


    - Local: 本地embedding + 本地database and 本地model
        - Generator: llama, gpt2 ... 
        - Embedding: sentence_transformer embedding 
----  

10. The general pipeline is ,先用 instruction-tuned LLM, 再加 RAG,有时再叠加 domain fine-tuning


----  


11. Multi-document reasoning 问题的答案不是在一个文档块里，而是要综合多个文档的内容。 
    - Multi chunks may need reorder, there may be conflict, need to reasoning. 
        - reranker
        - map-reduce summarize
        - chain-of-thought style synthesis
        - graph-based retrieval
    Template: f""" You are a helpful assistant.
            Answer the question only based on the context below.
            If the answer is not in the context, say you do not know.

            Context:
            {doc1}
            ...
            {doc2}
            ...

            Question:
            {query}
            ... """

----  

## General step 
It defines the similarity and apply on the documents to find the most related document. 


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# 1. 文档
documents = [
    "Gradient descent is an optimization algorithm used to minimize loss.",
    "Overfitting happens when a model memorizes training data.",
    "Regularization can reduce overfitting."
]

# 2. embedding model
model_name = "all-MiniLM-L6-v2" 
embed_model = SentenceTransformer(model_name) 

# 3. embedding the data doc
doc_embedding = embed_model.encode(documents)

# 4. embedding the query ,
# Note: the query have to be in the list formate 
query = "What is gradient descent"
query_embedding = embed_model.encode([query])[0] 

# 5. Evaluate the similarity 

def  cosine_similarity(query, doc_embedding): 
    
    return np.dot(query, doc_embedding)/ (np.linalg.norm(query)* np.linalg.norm(doc_embedding) ) 

# return [1, d] * [1, d] evaulte the scores for each documents  

scores = [ cosine_similarity( query, d) for d in doc_embedding ]

# 6. Define the top_k ,return the topk indices 
top_k = 2 
top_indices = np.argsort( scores)[::-1][:top_k] 
# the document and document embedding has the same index 
retrieved_docs = [ documents[i] for i in top_indices]                       

for doc in retrieved_docs: 
    print(doc)
# 7. Prompt 

# " join(list )"
context = "\n".join(retrieved_docs) 

prompt = f"""

Answer the question based only on the context below. 
context: 
{context}

question: 
{query} 

"""

# 8. Load the LLM 

 

KeyboardInterrupt: 

## Using the vector database 

In [ ]:

import chromadb 
# the sentence_transformers is different form the transformer 
from sentence_transformers import SentenceTransformer 


#Initilalize the chromadb object chromadb.Client()
client = chromadb.Client()  
collection = client.create_collection(name = 'docs') 

doc_embedding = SentenceTransformer.encode( documents )

#First the doc embedding should store in a list 
doc_embedding = doc_embedding.tolist() 
# add the embedding, document, and the metadata into chroma.Client(). create_collection( )
for i , doc in enumerate( documents):
    # inside is all list
    collection.add( 
                   ids = [str(i)],# index 
                   documents =[ documents[i]] , 
                   embeddings = [doc_embedding[i]], 
                   metadata = [ {"source": f"doc_{i}"}]
                   
                   )
    
# query embedding is also to list 
query_embedding = SentenceTransformer.encode( [query])[0]. tolist()


# This is the main difference , start tosearch fro mteh collection 
# use the collection.query 
results = collection.query( query_embeddings = [query_embedding], 
                            n_results = 2) 
# resutls has the documents key， and [0] represnets the first query. 
retreived_docs = results["documents"][0] 

context = "\n".join(retrieved_docs )




# Load local model , without use the chromadb, not use the appi 
 

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# 1. embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. local LLM
model_name = "distilgpt2"   # 演示用，小模型效果有限
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# 3. documents
documents = [
    "Gradient descent minimizes a loss function by iterative updates.",
    "Overfitting can be reduced using regularization and dropout.",
    "RAG combines retrieval with generation."
]

# 4. vector db
client = chromadb.Client()
collection = client.create_collection(name="rag_local")

# Here the embedded model dirrect work on the text? no need to do tokenizer?
doc_embeddings = embed_model.encode(documents).tolist()

for i, doc in enumerate(documents):
    collection.add(
        ids=[str(i)],
        documents=[doc],
        embeddings=[doc_embeddings[i]]
    )

# 5. query
query = "How do we reduce overfitting?"

query_embedding = embed_model.encode([query])[0].tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=2
)

retrieved_docs = results["documents"][0]
context = "\n".join(retrieved_docs)

prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{query}

Answer:
"""

# the tokenizer  is only for the prompt, but not the documents 
inputs = tokenizer(prompt, return_tensors="pt")

#  model.generate( input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"],
outputs = model.generate( **inputs, max_new_tokens = 80, do_sample = False )
# The output datasize is [batch_size, sequence_length] 
answer = tokenizer.decode(outputs[0], skip_special_tokens = True )
 
print(answer)  


# To check only the answers 
input_len = inputs['input_ids'].shape[1] # get the seq length 
newtoken = outputs[0][input_len:]
output_answer = model.decode( newtoken, skip_special_tokens= True)
print(output_answer)